# training-step-cycle — faded example 3: Full 5-call cycle with two separately passed parameters

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `training-step-cycle`. Running the beacon reports progress on the `PyTorch: Training step cycle` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Training step cycle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`training-step-cycle`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "training-step-cycle"
DD_SUBTOPIC = "PyTorch: Training step cycle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The optimizer is constructed with a list of tensors. All tensors in that list have their gradients updated when `optimizer.step()` is called. Passing `[a, b]` means both `a` and `b` are updated each step. The 5-call cycle is identical regardless of how many parameters you're optimizing.

## Faded exercise 3

Implement `faded3_two_params(n_steps=40)`. Build `a = t.tensor([1.0], requires_grad=True)` and `b = t.tensor([0.0], requires_grad=True)`. Optimizer is `t.optim.SGD([a, b], lr=0.05)`. Target: `y = 0.5 * x - 1.0` over `x = t.linspace(0, 2, 10)`. Run the 5-call cycle. The blank is the `optimizer.step()` call.

**Fill in:** The optimizer.step() call that updates both a and b using their computed gradients.

In [ ]:
import torch as t

def faded3_two_params(n_steps=40):
    t.manual_seed(13)
    a = t.tensor([1.0], requires_grad=True)
    b = t.tensor([0.0], requires_grad=True)
    optimizer = t.optim.SGD([a, b], lr=0.05)
    x = t.linspace(0, 2, 10)
    y_target = 0.5 * x - 1.0
    losses = []
    for _ in range(n_steps):
        pred = a * x + b
        loss = ((pred - y_target) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        raise NotImplementedError()  # TODO: The optimizer.step() call that updates both a and b using their computed gradients.
        optimizer.zero_grad()
    return a.detach().clone(), b.detach().clone(), losses


def _test():
    import torch as t
    a_fit, b_fit, losses = faded3_two_params(n_steps=40)
    # a should be near 0.5, b near -1.0
    assert abs(a_fit.item() - 0.5) < 0.3, f'a = {a_fit.item()}, expected ~0.5'
    assert abs(b_fit.item() - (-1.0)) < 0.5, f'b = {b_fit.item()}, expected ~-1.0'
    assert losses[0] > losses[-1]
    assert len(losses) == 40


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def faded3_two_params(n_steps=40):
    t.manual_seed(13)
    a = t.tensor([1.0], requires_grad=True)
    b = t.tensor([0.0], requires_grad=True)
    optimizer = t.optim.SGD([a, b], lr=0.05)
    x = t.linspace(0, 2, 10)
    y_target = 0.5 * x - 1.0
    losses = []
    for _ in range(n_steps):
        pred = a * x + b
        loss = ((pred - y_target) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return a.detach().clone(), b.detach().clone(), losses
```
</details>